# 12 — Validate MDRO mart

## Goal

CRE and MDR *P. aeruginosa* are summarized separately from the main susceptibility mart.

The denominator here is patient-level and based on the underlying base organism within each cohort. A patient with base *E. coli*, ESCE-R *E. coli*, and CRE *E. coli* still contributes once to the *E. coli* denominator. The phenotype numerator is counted independently from that base-organism denominator.

In [ ]:
%sql
select
    count(*) as mart_rows,

    count(
        distinct mdro_mart_key
    ) as unique_keys,

    sum(
        case
            when denominator_count
                 <> positive_count
                  + not_detected_count
                  + not_evaluable_count
                then 1
            else 0
        end
    ) as denominator_mismatches,

    sum(
        case
            when positive_count > denominator_count
                then 1
            else 0
        end
    ) as numerator_exceeds_denominator,

    min(phenotype_pct) as min_phenotype_pct,
    max(phenotype_pct) as max_phenotype_pct

from workspace.micro_dev.mart_mdro;

### Structural checks

I expect one unique mart row per source release, cohort, base organism, and MDRO phenotype. The positive, not-detected, and not-evaluable counts must add back to the denominator, and no numerator can exceed its denominator.

The observed phenotype percentages range from 0.0% to 13.4%, so there are no obvious out-of-range results.

In [2]:
%sql
with denominator_patient_groups as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,

        max(
            case
                when ast_cre_status <> 'not_applicable'
                    then 1
                else 0
            end
        ) as cre_eligible,

        max(
            case
                when ast_mdr_p_aeruginosa_status <> 'not_applicable'
                    then 1
                else 0
            end
        ) as mdr_p_aeruginosa_eligible

    from workspace.micro_dev.int_reporting_culture_organism

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group
),

expected_denominators as (

    select
        source_version,
        cohort_id,
        organism_reporting_group as organism_base_name,
        'CRE' as mdro_phenotype,
        count(*) as expected_denominator_count

    from denominator_patient_groups

    where cre_eligible = 1

    group by
        source_version,
        cohort_id,
        organism_reporting_group

    union all

    select
        source_version,
        cohort_id,
        organism_reporting_group,
        'MDR Pseudomonas aeruginosa',
        count(*)

    from denominator_patient_groups

    where mdr_p_aeruginosa_eligible = 1

    group by
        source_version,
        cohort_id,
        organism_reporting_group
),

expected_numerators as (

    select
        source_version,
        cohort_id,
        organism_reporting_group as organism_base_name,

        case
            when reporting_phenotype = 'cre'
                then 'CRE'

            when reporting_phenotype = 'mdr_p_aeruginosa'
                then 'MDR Pseudomonas aeruginosa'
        end as mdro_phenotype,

        count(*) as expected_positive_count

    from workspace.micro_dev.int_first_reporting_isolate

    where reporting_phenotype in (
        'cre',
        'mdr_p_aeruginosa'
    )

    group by
        source_version,
        cohort_id,
        organism_reporting_group,
        reporting_phenotype
),

comparison as (

    select
        m.source_version,
        m.cohort_id,
        m.organism_base_name,
        m.mdro_phenotype,

        m.denominator_count,
        d.expected_denominator_count,

        m.positive_count,
        coalesce(
            n.expected_positive_count,
            0
        ) as expected_positive_count,

        m.not_detected_count,
        m.not_evaluable_count,
        m.phenotype_pct,

        case
            when m.denominator_count
                 <> d.expected_denominator_count
                then 'denominator_mismatch'

            when m.positive_count
                 <> coalesce(
                     n.expected_positive_count,
                     0
                 )
                then 'numerator_mismatch'

            else 'pass'
        end as validation_status

    from workspace.micro_dev.mart_mdro m

    left join expected_denominators d
        on m.source_version = d.source_version
       and m.cohort_id = d.cohort_id
       and m.organism_base_name = d.organism_base_name
       and m.mdro_phenotype = d.mdro_phenotype

    left join expected_numerators n
        on m.source_version = n.source_version
       and m.cohort_id = n.cohort_id
       and m.organism_base_name = n.organism_base_name
       and m.mdro_phenotype = n.mdro_phenotype

)

select
    validation_status,
    count(*) as mart_rows

from comparison

group by
    validation_status

order by
    validation_status

,validation_status,mart_rows
0,pass,31


### Independent numerator and denominator check

This query rebuilds the patient-level denominators from `int_reporting_culture_organism` and the positive numerators from `int_first_reporting_isolate`, then compares them with `mart_mdro`.

All 31 mart rows pass.

### Takeaway

Both final reporting marts now reconcile independently to the intermediate models. At this point the phenotype-aware pipeline is validated end to end and the remaining work is presentation: documentation, Tableau, and repository cleanup.